# Context Selection

**Prerequisites:** `01_context_basics.ipynb`. From `04_memory`: `03_operations.ipynb` (for `Recall`)

**Selection decides which of the information already in hand goes into one call. It can only leave things out.**

The other way information gets in is **retrieval**: asking a memory store a question and getting back things that were *not* in hand. The two are easy to blur, because both end with "these items go into the context". They start from different places and fail differently:

```
what the run holds ───→ select   ──┐
                                    ├──→ available context ──→ construct (notebook 4)
memory store ─────────→ retrieve ──┘
```

Selection works over a list you can see in full, so its mistakes are *omissions*: something you had was left out. Retrieval works against a store too big to look at, so it can also *add* the wrong thing. Neither needs the other's output, so they run side by side, and this notebook ends by running them at the same time.

`select()` works in a fixed order, and the order is the design:

1. **supersede**: of several versions of one thing, keep the newest
2. **pin**: preserved items always go in
3. **score**: rank the rest by relevance, importance and recency
4. **fill**: best first, until the budget is spent

## Setup

In [1]:
import shutil
import sys
import tempfile
import time
from pathlib import Path

HERE = Path.cwd().parent
sys.path.append(str(HERE))
for chapter in ("01_foundations", "02_agent_runtime", "04_memory"):
    sys.path.append(str(HERE.parent / chapter))

import stores
from context import available, render, tokens
from incident import RULES, incident
from llm import DEFAULT_MODEL, client
from recall import Recall
from selection import IMPORTANCE, gather, overlap, retrieved, select, supersede
from stores import Journal, cosine, embed


def ask(items):
    # the same stand-in for construction as notebook 1 -- instructions as the system message,
    # everything else one labelled line per item. Notebook 4 replaces it.
    system = "\n".join(i.content for i in items if i.kind == "instruction")
    rest = [i for i in items if i.kind != "instruction"]
    response = client.chat.completions.create(model=DEFAULT_MODEL, messages=[
        {"role": "system", "content": system},
        {"role": "user", "content": render(rest)},
    ])
    print(f"prompt {response.usage.prompt_tokens} tokens\n")
    print(response.choices[0].message.content)
    return response

## What there is to choose from

The session from notebook 1, as `available()` lists it: every item, the kind of thing it is, what it is *about* if it is one version of something, and its size.

In [2]:
conversation, pad = incident()
items = available(conversation, pad)

for i in items:
    print(f"t{i.turn:<3}{i.kind:<12}{i.tokens:>5}  {i.about[:38]:<38}  {' '.join(i.content.split())[:48]}")

by_kind = {}
for i in items:
    by_kind[i.kind] = by_kind.get(i.kind, 0) + i.tokens
print(f"\n{tokens(items)} tokens:", ", ".join(f"{k} {v}" for k, v in sorted(by_kind.items(), key=lambda kv: -kv[1])))

t0  instruction    11                                          You are an on-call assistant helping investigate
t1  message        49                                          user: Checkout has been slow since this morning.
t2  state          21  plan                                    Goals: 0. Check checkout-api latency 1. Read che
t4  observation    21  get_metrics {"service": "checkout-api"  checkout-api p95 latency is 2.4 s, up from 0.3 s
t4  tool_result   145  get_metrics {"service": "checkout-api"  { "service": "checkout-api", "window": "last 15m
t6  observation    30  get_logs {"service": "checkout-api", "  The checkout-api logs since 07:00 show repeated 
t6  tool_result  1142  get_logs {"service": "checkout-api", "  07:12:24 INFO checkout-api POST /checkout 200 20
t8  observation    32  get_deploys {"service": "checkout-api"  The last checkout-api deploy was v4.12.0 at 07:0
t8  tool_result   119  get_deploys {"service": "checkout-api"  [ { "version": "v4.12.0", "at": "07:05", 

Raw tool results are four fifths of everything, and one of them, the logs, is half on its own. Every call appears twice: once as `Observe`'s sentence (`observation`) and once as what the tool really returned (`tool_result`). Both share an `about`, the tool and its arguments, so `get_metrics {"service": "checkout-api"}` at t4 and t19 are marked as two versions of the same reading.

## Recency alone

The simplest selection there is: keep the most recent things that fit. It's what chapter 1's `trim_to_budget()` does with messages, and it's a sensible default. What happened last is usually what matters next.

Here it is on items, with a budget of 250 tokens and every weight except recency set to zero:

In [3]:
recent = select(items, budget=250, w_relevance=0, w_importance=0, w_recency=1)
print(recent.report())

kept 10 items, 249 tokens
  + pinned t0   instruction You are an on-call assistant helping investigate production incidents.
  +        t14  message     user: Ok. Could it be the database itself? Check orders-db.
  +        t15  state       Replanning: the database has to be ruled out before blaming the dep...
  +        t17  observation orders-db CPU is 22% and its query p95 is 4 ms; it is healthy.
  +        t17  tool_result { "service": "orders-db", "cpu_percent": 22, "query_latency_ms": { ...
  +        t19  observation checkout-api p95 latency is now 2.9 s.
  +        t20  message     assistant: orders-db is healthy (query p95 4 ms), so the database i...
  + pinned t21  request     So what is causing the checkout slowdown, how bad is it now, and wh...
  +        t22  state       suspect: deploy v4.12.0 (orders-db pool 50 -> 5)
  +        t22  state       ruled_out: orders-db, search-api
dropped 14
  -        t1   message     user: Checkout has been slow since this morning...  <- o

The most recent quarter of the session made it in. So did the scratchpad, which counts as recent because it describes where the task stands now. The system message and the request are pinned, which `available()` does for them. Three items were dropped as superseded, the old plan and the old checkout-api reading twice over, which is a later section's subject.

**The t1 message was dropped for being old.** That message holds both rules: milliseconds, and read-only. The user said them once, at the start, meaning "for this whole session". A recency score reads that as "a long time ago". Asked the question now:

In [4]:
recent_answer = ask(recent.kept)

prompt 349 tokens

The checkout‑api is now hitting **~2.9 s p95 latency**, which is a clear degradation (it was ~1 s before). The orders‑db itself is healthy (22 % CPU, 4 ms p95 query latency, plenty of connections), so the bottleneck must be internal to checkout‑api.

The most likely cause is **deploy v4.12.0**, which changed the DB connection pool from 50 to 5. With only five connections the service is queuing requests, inflating the request‑time even though the database can serve queries instantly.

**What to do**

1. **Rollback** the pool change (or redeploy v4.12.0 with the pool size restored to ≈ 50).  
2. If a rollback isn’t possible immediately, hot‑patch the config to increase the pool size and restart checkout‑api.  
3. After the change, re‑measure checkout‑api latency; it should drop back toward the pre‑incident p95 (~1 s).  

In short: the slowdown is caused by an undersized DB connection pool introduced in the recent deploy; restore the pool size to resolve it.


Three things went wrong, and each one traces back to an item the selection dropped:

- **"~2.9 s"**: in seconds. The milliseconds rule was in t1.
- **"restart checkout-api"**, and a redeploy: recommended as the fix. The read-only rule was in t1 too.
- **"it was ~1 s before"**: invented. The real baseline, 0.3 s, is in the t4 reading, which was superseded. Asked *how bad*, with nothing to compare against, the model supplied a comparison.

Nothing in the answer signals any of this. It's fluent, confident, and mostly right, which is exactly what makes a dropped item hard to notice.

## Preserved facts and constraints

**A constraint isn't old news. It's a standing instruction, and it has to reach every call it applies to, however long ago it was said.**

Nothing about the t1 message says so. Its role is `user`, the same as small talk. Something has to *mark* it as a rule, and that is a decision made outside selection: here the notebook's author makes it by passing `constraints=RULES`. In a live system something has to notice a rule when it's said and write it down. That's *fact extraction*, a compression operation, and notebook 3 builds it.

Once marked, a constraint is **pinned**. It goes into every selection, ahead of any scoring.

In [4]:
items = available(conversation, pad, constraints=RULES)
pinned_recent = select(items, budget=250, w_relevance=0, w_importance=0, w_recency=1)
print(pinned_recent.report())

kept 11 items, 237 tokens
  + pinned t0   instruction You are an on-call assistant helping investigate production incidents.
  + pinned t0   constraint  Everything you do must be read-only: do not restart, scale or redep...
  + pinned t0   constraint  Give every duration in milliseconds.
  +        t14  message     user: Ok. Could it be the database itself? Check orders-db.
  +        t17  observation orders-db CPU is 22% and its query p95 is 4 ms; it is healthy.
  +        t17  tool_result { "service": "orders-db", "cpu_percent": 22, "query_latency_ms": { ...
  +        t19  observation checkout-api p95 latency is now 2.9 s.
  +        t20  message     assistant: orders-db is healthy (query p95 4 ms), so the database i...
  + pinned t21  request     So what is causing the checkout slowdown, how bad is it now, and wh...
  +        t22  state       suspect: deploy v4.12.0 (orders-db pool 50 -> 5)
  +        t22  state       ruled_out: orders-db, search-api
dropped 15
  -        t1   mes

Both rules are in, and at the same budget. That's not free: pinned items are paid for first, and something else came out to make room. Here it was the revised plan at t15. Pinning is a promise, so use it sparingly. Pin everything and there is nothing left to select.

It also means a budget can be too small to keep the promise. When that happens `select()` refuses instead of quietly dropping a pinned item:

In [5]:
try:
    select(items, budget=50)
except ValueError as e:
    print(e)

pinned items alone are 65 tokens, over the budget of 50. Selection can leave things out; it cannot make what must go in any smaller. Raise the budget or compress them.


Selection can leave things out. It can't make what must go in any smaller. That's what compression is for.

## Superseded: tool results and task state

**Some items aren't less relevant; they're wrong now.** The plan at t2 was replaced at t15. The checkout-api reading at t4 (2.4 s) was replaced by the one at t19 (2.9 s). A scratchpad key written twice would be the same. These are versions of one thing, and only the newest is current.

That's a question of *identity*, and `select()` settles it before any scoring, by `about`:

In [6]:
current, old = supersede(items)
for item, why in old:
    print(f"t{item.turn:<3}{item.kind:<12}{why:<16}  {' '.join(item.content.split())[:60]}")

t2  state       superseded at t15  Goals: 0. Check checkout-api latency 1. Read checkout-api lo
t4  observation superseded at t19  checkout-api p95 latency is 2.4 s, up from 0.3 s yesterday.
t4  tool_result superseded at t19  { "service": "checkout-api", "window": "last 15m", "latency_


Why not leave it to scoring? Recency would usually rank the new reading above the old one. But "usually" and "above" aren't enough: with any budget big enough for both, both get in, and the model is handed two answers to "how slow is checkout" and has to work out which is current, typically from position alone.

Relevance is worse than no help here. Scored by meaning against the request, the **stale** reading beats the current one. `embed()` is chapter 4's Gemini call:

In [7]:
def meaning(query, text):
    return cosine(embed([query], task="RETRIEVAL_QUERY")[0], embed([text])[0])

request = next(i.content for i in items if i.kind == "request")
at = {(i.kind, i.turn): i for i in items}
for turn in (4, 19):
    item = at["observation", turn]
    print(f"t{turn:<3} {meaning(request, item.content):.3f}  {item.content}")

t4   0.778  checkout-api p95 latency is 2.4 s, up from 0.3 s yesterday.


t19  0.741  checkout-api p95 latency is now 2.9 s.


The old reading says *"up from 0.3 s yesterday"*, which reads more like an answer to "how bad is it" than *"is now 2.9 s"* does. Relevance measures resemblance to the question, not whether something is still true. Chapter 4 reached the same conclusion about memory: whether a fact still holds is settled by *supersession*, never by nudging a stale one down a ranking.

## Observations and tool results

Every tool call left two items, and they aren't the same information at different lengths. They're different information:

In [8]:
for turn in (4, 6, 8, 12, 17, 19):
    o, r = at["observation", turn], at["tool_result", turn]
    print(f"t{turn:<3}{o.about[:40]:<40} observation {o.tokens:>3}   tool_result {r.tokens:>5}")

t4  get_metrics {"service": "checkout-api"}  observation  21   tool_result   145
t6  get_logs {"service": "checkout-api", "si observation  30   tool_result  1142
t8  get_deploys {"service": "checkout-api"}  observation  32   tool_result   119
t12 get_metrics {"service": "search-api"}    observation  14   tool_result   143
t17 get_metrics {"service": "orders-db"}     observation  21   tool_result    57
t19 get_metrics {"service": "checkout-api"}  observation  13   tool_result   146


The observation is `Observe`'s one sentence about the result: small, already phrased, and it's what chapter 2 decided the agent should reason over. The tool result is what the tool actually returned: exact, often huge, and full of things the sentence left out. The t19 sentence says latency; the raw result also has the 3.4% error rate, the p99, and the request rate.

Which one a call needs depends on the call. A `Think` answering "what's causing this" is well served by sentences. An operation *checking* a number, or answering "how bad is it" in figures, needs the raw result. `kinds=` restricts what an operation can see at all, before anything is scored:

In [9]:
checking = select(items, budget=400, query="error rate and p99 latency now",
                  kinds=("tool_result",))
print(checking.report(width=60))

kept 6 items, 354 tokens
  + pinned t0   instruction You are an on-call assistant helping investigate producti...
  + pinned t0   constraint  Everything you do must be read-only: do not restart, scal...
  + pinned t0   constraint  Give every duration in milliseconds.
  +        t12  tool_result { "service": "search-api", "window": "last 15m", "latency...
  +        t19  tool_result { "service": "checkout-api", "window": "last 15m", "laten...
  + pinned t21  request     So what is causing the checkout slowdown, how bad is it n...
dropped 20
  -        t1   message     user: Checkout has been slow since th...  <- kind message not used here
  -        t2   state       Goals: 0. Check checkout-api latency ...  <- superseded at t15
  -        t4   observation checkout-api p95 latency is 2.4 s, up...  <- superseded at t19
  -        t4   tool_result { "service": "checkout-api", "window"...  <- superseded at t19
  -        t6   observation The checkout-api logs since 07:00 sho...  <- kind obs

Pinned items still go in: the rules apply to a checker too. Everything else is raw results only. The current checkout-api metrics (t19) are in. So are **search-api's** (t12), which have nothing to do with the question: every metrics blob contains `error_rate` and `p99`, so by shared words they all look equally relevant. The logs at t6 (1,142 tokens) can never fit a 400-token budget, whatever they score. Anything that needs something *from* the logs needs them made smaller first.

## Relevance

**Relevance is how much an item has to do with what this call is for.** `select()` takes it as a function, `relevance(query, text) -> float`, and the default is the cheapest one there is: the share of the query's words that appear in the item.

In [10]:
q = "Is the database the bottleneck?"
for i in items:
    if i.turn in (14, 15, 17, 20) and i.kind != "tool_result":
        print(f"t{i.turn:<3}{i.kind:<12}overlap {overlap(q, i.content):.2f}   {' '.join(i.content.split())[:60]}")

t14 message     overlap 0.50   user: Ok. Could it be the database itself? Check orders-db.
t15 state       overlap 0.50   Replanning: the database has to be ruled out before blaming 
t17 observation overlap 0.00   orders-db CPU is 22% and its query p95 is 4 ms; it is health
t20 message     overlap 1.00   assistant: orders-db is healthy (query p95 4 ms), so the dat


The item that *answers* the question, t17, where the tool reported orders-db's CPU and query time, scores **zero**. It never says "database" or "bottleneck"; it says `orders-db`, `CPU`, `healthy`. The items that score highest are the ones that *talk about* the question: the user asking it, and the assistant's summary.

Word overlap measures shared vocabulary, and evidence rarely shares vocabulary with the question it settles. Swapping in meaning, chapter 4's embeddings, is one argument:

In [11]:
# one batch call up front; embed() caches, so meaning() below never calls Gemini per item
embed([i.content for i in items])

for i in items:
    if i.turn in (14, 15, 17, 20) and i.kind != "tool_result":
        print(f"t{i.turn:<3}{i.kind:<12}meaning {meaning(q, i.content):.3f}   {' '.join(i.content.split())[:60]}")

t14 message     meaning 0.697   user: Ok. Could it be the database itself? Check orders-db.
t15 state       meaning 0.715   Replanning: the database has to be ruled out before blaming 
t17 observation meaning 0.694   orders-db CPU is 22% and its query p95 is 4 ms; it is health
t20 message     meaning 0.745   assistant: orders-db is healthy (query p95 4 ms), so the dat


Now t17 scores close to the others instead of zero. The ordering still favours the items that restate the question, but the evidence is back in contention.

What it costs: an embedding for every candidate, a network call to a second provider. The first version of this cell embedded one item at a time, and scoring this session took about eighteen seconds. Batching them into one request up front, as above, is what makes it usable. Embeddings are deterministic, so the cache is safe.

## Importance and recency

**Importance is a property of the *kind* of item, fixed in advance.** Not rated per item by a model: chapter 4 measured that, and a model asked "how important is this, 1–10" answers 6 to 8 for nearly everything, one call per item for a number that reorders nothing.

In [12]:
for kind, weight in sorted(IMPORTANCE.items(), key=lambda kv: -kv[1]):
    print(f"{kind:<12}{weight}")

instruction 1.0
request     1.0
constraint  1.0
state       0.8
decision    0.8
fact        0.7
summary     0.7
observation 0.6
memory      0.6
message     0.4
tool_result 0.3


Task state ranks high because it's where the next step starts from. A raw tool result ranks lowest, not because it doesn't matter, but because it's large and the observation usually says what matters in it. When it doesn't, `kinds=` is the tool, not the weight.

**Recency** is `0.5 ** (age / half_life)`, with age counted in turns. At the default `half_life=8`, the t1 message is worth about a sixth of the latest. That's right for chatter and exactly wrong for a rule, which is why rules are pinned and not scored.

The three terms are min-max normalised across the candidates before they're weighted, as in chapter 4's `Recall`, so a weight is a share of the decision and not a share of whatever range a term happens to occupy.

## The budget

With everything in place, the budget decides how far down the ranking the fill goes. Three budgets, word overlap for relevance:

In [13]:
for budget in (200, 400, 800):
    s = select(items, budget=budget)
    kept = [f"t{i.turn}:{i.kind[:4]}" for i in s.kept if not i.pinned]
    print(f"budget {budget:>4} -> {s.tokens:>4} tokens  {' '.join(kept)}")

budget  200 ->  196 tokens  t12:obse t15:stat t19:obse t20:mess t22:stat t22:stat
budget  400 ->  390 tokens  t6:obse t8:obse t15:stat t19:obse t19:tool t20:mess t22:stat t22:stat
budget  800 ->  771 tokens  t1:mess t6:obse t8:obse t9:mess t10:mess t12:obse t12:tool t13:mess t14:mess t15:stat t17:obse t17:tool t19:obse t19:tool t20:mess t22:stat t22:stat


At **200** tokens the deploy observation (t8), the one that names the cause, is out, and the search-api detour (t12) is in. Two things combine. Word overlap barely separates these items: almost everything mentions "checkout", so almost everything scores the same. And the fill is greedy: once the top-scoring items are in, t8 (32 tokens) no longer fits and t12 (14 tokens) still does. Leftover space goes to whatever is small enough, however low it scored. A floor on the score, like the similarity floor on chapter 4's `Recall`, would leave that space empty instead. `select()` doesn't have one.

The cause still reaches the call at 200, through the scratchpad's `suspect` entry. That's why task state is weighted high: it's the run's own summary of where things stand, and it's small.

At **400** the logs' observation and the deploy observation are in, along with the current raw metrics. At **800** nearly everything is in except the raw logs, which no budget here can hold.

## Recency is a weight, not a promise

`w_recency` makes recent items score higher. It does not keep them. An older item with strong relevance outranks a recent one, and the greedy fill can skip a recent item that does not fit while taking a smaller older one. "The last few turns are always in context" is a reasonable thing to want, and no combination of weights provides it.

`keep_last=N` does: the N most recent items join the pinned ones as must-keeps, before any scoring happens.

In [14]:
newest = sorted(items, key=lambda i: i.turn)[-5:]
print("the 5 newest items:", [f"{i.kind}@{i.turn}" for i in newest], "\n")

for label, kept in (("scored, budget 300", select(items, budget=300).kept),
                    ("keep_last=5, budget 300", select(items, budget=300, keep_last=5).kept)):
    lost = [f"{i.kind}@{i.turn}" for i in newest if not any(k is i for k in kept)]
    print(f"{label:<24}{'all 5 present' if not lost else 'missing ' + str(lost)}")

the 5 newest items: ['tool_result@19', 'message@20', 'request@21', 'state@22', 'state@22'] 

scored, budget 300      missing ['tool_result@19']
keep_last=5, budget 300 all 5 present


Scoring dropped `tool_result@19` — the current raw metrics. That is the item carrying the 3.4% error rate, and it is the *newest* tool result in the run. It went out because a raw result is the lowest-importance kind (0.3) and 146 tokens is expensive, so three smaller older items beat it.

That exact absence is what made the model invent figures in notebook 1 and again in notebook 3. Asked "how bad is it now" with the current measurements missing, it answers anyway.

`keep_last` is the guarantee, and it is a guarantee with teeth: like the pinned items, if the last N do not fit the budget, selection refuses rather than silently dropping one.

In [15]:
try:
    select(items, budget=120, keep_last=5)
except ValueError as e:
    print(e)

pinned items and the last 5 alone are 277 tokens, over the budget of 120. Selection can leave things out; it cannot make what must go in any smaller. Raise the budget or compress them.


## Selection is not retrieval

Everything so far has chosen among things the run already held. Some of what this call should know was never in the session at all. This has happened before: in June, checkout slowed down after a config change shrank the same pool, and the fix is written down in a runbook. That's memory, and getting it into context is a **retrieve**: chapter 4's `Journal`, ranked by its `Recall`.

The journal gets a few past write-ups in a fresh directory:

In [16]:
store = Path(tempfile.mkdtemp())
journal = Journal(store)
journal.write([
    ("2026-06-03", "checkout-api", "postmortem",
     "Checkout latency spiked to 3 s after a config change shrank the orders-db connection pool. "
     "Raising the pool back to 50 in the config service fixed it within two minutes; nothing was redeployed."),
    ("2026-07-14", "orders-db", "postmortem",
     "orders-db failover at 14:02 caused 90 seconds of checkout errors; the replica was promoted cleanly."),
    ("2026-08-02", "search-api", "note",
     "search-api p95 regularly reaches 200 ms during the nightly reindex; this is expected."),
    ("2026-08-20", "checkout-api", "runbook",
     "checkout-api's orders-db pool size is set in the config service under "
     "checkout/orders_db/pool_size and takes effect without a restart."),
    ("2026-09-01", "payments", "postmortem",
     "payments-sdk 3.2.0 leaked sockets under load; 3.2.1 fixed it."),
])
recall = Recall(journal, today="2026-09-21")

for m in retrieved(recall, request, k=3):
    print(f"{m.source:<11} {m.content}")

journal#1   #1  2026-06-03  Checkout latency spiked to 3 s after a config change shrank the orders-db connection pool. Raising the pool back to 50 in the config service fixed it within two minutes; nothing was redeployed.
journal#2   #2  2026-07-14  orders-db failover at 14:02 caused 90 seconds of checkout errors; the replica was promoted cleanly.
journal#5   #5  2026-09-01  payments-sdk 3.2.0 leaked sockets under load; 3.2.1 fixed it.


The top hit is exactly right: the June incident, the same pool, and how it was fixed without a redeploy. Nothing in the session could have supplied that, however good the selection.

The other two are wrong. A failover and an SDK leak cleared `Recall`'s similarity floor because they're about checkout, errors and load. And the runbook, which holds the exact config key, didn't come back at all: the request asks about cause and severity, and the runbook is about configuration. (The floor, 0.6, was measured on chapter 4's journal, not this one.)

This is the failure selection can't have. **Selection can only drop things you had. Retrieval can also bring in things that don't belong, and miss the one that does.** It's also why retrieval's results are items like any other: they can be selected from, and the budget still applies.

### What gets injected depends on the store

Those entries are short, so injecting them whole is fine. Real write-ups are not, and `04_memory` settles the rule: **a journal that writes summaries has a second stage**, so a retrieval injects the *summary* and lets the model call `open_memory(id)` for a body it decides it needs.

Getting that wrong in the generous direction is the expensive mistake, and it hides: injecting full bodies means the model already has everything, so it never opens anything, two-stage recall silently stops happening, and the answers stay correct while the prompt triples. Chapter 4 measured 2,136 characters against 363 for the same three memories.

A journal with longer write-ups and a summarizer, so there is something to be lossy about:

In [17]:
long_store = Path(tempfile.mkdtemp())

def first_sentences(text, keep=2):
    # stands in for chapter 4's model-written summaries: cheap, deterministic, good enough to
    # show the footprint. A real summary is a model call on the WRITE path, paid once per memory.
    return " ".join(text.split(". ")[:keep]).rstrip(".") + "."

long_journal = Journal(long_store, summarize=first_sentences)
long_journal.write([
    ("2026-06-03", "checkout-api", "postmortem",
     "Checkout latency spiked to 3 s at 09:14 after a routine config change shrank the orders-db "
     "connection pool from 50 to 5. Raising the pool back to 50 in the config service fixed it "
     "within two minutes and nothing was redeployed. The change had been reviewed by two people "
     "and neither noticed the pool field, because the diff was mostly unrelated timeout tuning. "
     "We added a guard that rejects a pool size below 20 for any service handling over 50 "
     "requests per second, and a dashboard panel showing pool saturation next to latency. The "
     "incident lasted 41 minutes end to end, of which 33 were spent looking at the database."),
    ("2026-08-20", "checkout-api", "runbook",
     "checkout-api's orders-db pool size is set in the config service under "
     "checkout/orders_db/pool_size and takes effect without a restart. Values are picked up "
     "within about 15 seconds by every instance. Do not edit the value in the deploy manifest: "
     "the config service wins, so a manifest edit looks applied and changes nothing. If the "
     "config service is unreachable, instances keep the last value they saw."),
])
long_recall = Recall(long_journal, today="2026-09-25")

two_stage = retrieved(long_recall, request, k=2)
for m in two_stage:
    print(f"[{m.kind:<11}] {m.content[:110]}")

bodies = sum(len(long_journal.get(int(m.about.split("#")[1]))["text"])
             for m in two_stage if m.kind == "memory")
print(f"\ninjected: {sum(len(m.content) for m in two_stage)} chars   "
      f"the same memories in full: {bodies} chars")

[memory     ] #1  2026-06-03  Checkout latency spiked to 3 s at 09:14 after a routine config change shrank the orders-db con
[memory     ] #2  2026-08-20  checkout-api's orders-db pool size is set in the config service under checkout/orders_db/pool_
[instruction] The memories above are summaries. Call open_memory(memory_id=N) for the full text of one before answering from

injected: 568 chars   the same memories in full: 1019 chars


Here that is 568 characters against 1,019 — a narrower gap than chapter 4's, because these write-ups are shorter and the stand-in summarizer keeps two sentences rather than writing one. The shape is what matters: the gap grows with the length of what is being summarised, and a real journal's entries are pages.

The id is the part that matters, and it is in the **content**, not only in the item's `about`. That is a deliberate change: an id carried only as a label disappears the moment anything renders without labels, and `04_memory` measured ids surviving one live run in three once a listing had been through `Observe`'s paraphrase. Losing the id does not make the answer worse — it removes the second stage, because there is nothing left to open.

So it has to survive `flat()`, `build()` with labels off, and the rebuild into runtime messages that notebook 5 uses:

In [18]:
from construction import build, flat
from focus import to_messages

for name, text in (("flat()", flat(two_stage)[0]["content"]),
                   ("build(), labels off", build(two_stage)[1]["content"]),
                   ("to_messages()", str(to_messages(two_stage)))):
    print(f"  '#{two_stage[0].about.split('#')[1]}' survives {name:<20}",
          two_stage[0].content.split()[0] in text)

  '#1' survives flat()               True
  '#1' survives build(), labels off  True
  '#1' survives to_messages()        True


## Running both at once

Selection and retrieval don't need each other's output. Selection needs the session and the request; retrieval needs the store and the request. So there's no reason to wait for one before starting the other. `gather()` runs one selection and any number of retrievals at the same time, each given as a zero-argument function, so every detail of *how* each runs stays with the caller.

Both halves here go over the network, because selection scores by meaning. To make the timing honest, the embedding cache is emptied before each run, so neither version gets the other's work for free.

In [19]:
def selecting():
    embed([i.content for i in items])              # batch, as above
    return select(items, budget=400, relevance=meaning)

def retrieving():
    return retrieved(recall, request, k=2)

stores._EMBED_CACHE.clear()
t = time.perf_counter()
selection, memories = selecting(), retrieving()
one_after_other = time.perf_counter() - t

stores._EMBED_CACHE.clear()
t = time.perf_counter()
selection, memories = gather(selecting, retrieving)
side_by_side = time.perf_counter() - t

print(f"one after the other: {one_after_other:.2f}s")
print(f"side by side       : {side_by_side:.2f}s")

one after the other: 3.01s
side by side       : 1.85s


Side by side, the pair takes about as long as the slower of the two instead of their sum. With one retrieval that's a modest saving. It grows with every store an agent consults, because each extra retrieval adds nothing to the wait unless it's the slowest.

What comes out is two lists: what was selected from the run, and what was retrieved from memory. Together they're the *available context* for this call, and each had its own budget: 400 tokens for selection, two memories for retrieval. Nothing yet decides where each item goes in the model's input, which is construction's job in notebook 4. For now, the same stand-in:

In [20]:
final = ask(selection.kept + memories)

prompt 610 tokens

**Root cause** – The slowdown is coming from the checkout‑api itself, not the orders‑db. After the 07:05 deploy (v4.12.0) the orders‑db connection pool was reduced from **50** connections to **5**. The logs show “connection pool exhausted” errors starting at 07:12, which forces checkout requests to wait for a free connection, inflating latency.

**Current impact** – Checkout‑api 95th‑percentile latency is now **2 900 ms** (up from ~2 400 ms earlier and well above the normal ~200 ms range). The high‑latency tail is driven by the pool‑exhaustion bottleneck.

**Recommended fix (read‑only actions)**  
1. **Restore the pool size** to its previous value (≈ 50) via the configuration service; this is a non‑deploy change and can be applied instantly.  
2. If a larger pool isn’t viable, **throttle incoming checkout traffic** or implement a retry/back‑off strategy in the checkout service to prevent request queuing while waiting for a connection.  
3. Add **monitoring/alerts** o

588 prompt tokens, against 2,347 for everything in notebook 1. Durations are in milliseconds. The fix comes from memory, *"in the config service... nothing was redeployed"*, which is what the read-only rule wants. The diagnosis cites the log evidence.

One thing is wrong: *"latency should drop back to normal (≈ 180 ms for related services)"*. 180 ms is **search-api's** figure. Two selection decisions combined to put it there:

- The real baseline, "up from 0.3 s yesterday", lived only in the t4 reading, and t4 was superseded **as a whole item**. Its value was stale. The baseline inside the same sentence wasn't, and nothing else carried it.
- The search-api observation (t12) was kept, but the question that framed it, t10's *"Unrelated, but..."*, was dropped. Without it, "180 ms, which is normal" reads like a statement about checkout.

Both answers in this notebook invented a baseline, with a different number each time. The lesson isn't about baselines:

- **Selection works on items, and facts are smaller than items.** Dropping an item because one fact in it was replaced loses every other fact it carried. Pulling facts out of an item *before* it's dropped is fact extraction, in notebook 3.
- **Some items only mean what they mean next to another item.** An observation without the question that prompted it can say something different. Keeping related items together, and labelling what each one is, is construction, in notebook 4.

In [21]:
shutil.rmtree(store, ignore_errors=True)        # the journals were only ever for
shutil.rmtree(long_store, ignore_errors=True)   # this notebook

## What you built

`selection.py`:

- **`select(items, budget, query, kinds, relevance, ...)`**: supersede, pin, score, fill, in that order. Returns a `Selection` holding what was kept, in its original order, and what was dropped, each with the reason. The report is how every result in this notebook was read.
- **`supersede(items)`**: the identity step. Versions of one thing, found by `about`; only the newest survives. Not a score, because an old plan isn't less relevant than the new one. It's wrong.
- **`overlap`** as the default relevance, and any `(query, text) -> float` in its place. Here, chapter 4's embeddings, batched.
- **`IMPORTANCE`**: by kind, fixed in advance, never rated by a model.
- **`retrieved(recall, query)`**: a chapter 4 `Recall`'s hits as `memory` items. It does no ranking of its own, because `Recall` already did.
- **`gather(select_fn, *retrieve_fns)`**: selection and retrieval started together, because neither needs the other.

What selection can't do sets up the next notebook. When the rules alone overflowed the budget, `select()` refused. When the logs couldn't fit any budget, they were dropped whole, along with everything in them. Both need something that makes information **smaller** instead of choosing between it: **compression**.